In [ ]:
# TODO: informações cstfoil, XFOIL, e prints de single_run.py
# TODO: alterar ftol para 1e-5
# TODO: alterar batentes Al_lower, Al_upper, Au_lower, Au_upper

# Lab 3: Projeto de Perfil

Módulos:
1. eulerblock.exe: airfoil.dat, settings.txt -> wall.dat, solution.vtk, derivatives.dat
2. gridgen2d.py: gera a malha
3. gridwarp.py: deforma a malha mas acho que não faz nada agora
4. euler_mod.py: chama eulerblock.exe
5. airfoil_mod.py (modificado para não usar pip): gera airfoil.dat dos parâmetros

Scripts em eulerblock_run_cases:
1. 01_airfoil_plotter/airfoil_plotter.py: analisar airfoil interativamente
2. 02_single_run/single_run.py: analisar airfoil não-interativamente
3. 03_optimize/optimize.py: otimizar airfoil para uma condição. Pode reinicializar com reload_opt = opt_results.pickle. Parâmetros de otimização: ftol, batentes Al_lower, Al_upper, Au_lower, Au_upper
4. 03_optimize/plot_history.py: plota progresso da otimização
5. paraview_layouts/paraview_state_windows.py: visualizar solution.vtk no paraview

Objetivo: otimizar 2 perfis, um em y=0 e outro em y = 0.9*(b_w/2), para o cruzeiro:
1. Mach = mach_cruise
2. altitude = altitude_cruise
3. Cl = Cl(y): curva elíptica de sustentação no cruzeiro, deformada pela corda variável da asa trapezoidal
4. clmax(y) calculado no XFOIL > airplane["inputs"]["clmax_w"]

xfoil é usado como restrição na otimização e para gerar figuras automaticamente.

Avião usado: Tomav final de PRJ-22

Aerofólio inicial: NACA 1411

airfoil.dat -> XFOIL: utilize os seguintes comandos para ajustar a geometria:

• load airfoil.dat Carrega o aerofólio

• ppar Menu para ajuste dos painéis

• t Ajustar concentração de painéis no bordo de fuga

• 0.45 Fator de concentração

• n Ajustar número de painéis

• 200 Número de painéis

• <ENTER> Retornar ao menu anterior

• <ENTER> Retornar ao menu anterior

• oper Abrir menu de análise

• iter Definir número de iterações viscosas

• 100 Número máximo de iterações viscosas

• v Ativas análise viscosa

• Re_a ou Re_b Coloque o número de Reynolds desejado

• a Analisar ângulo de ataque

• 5 Angulo de ataque desejado (em graus) 

## Google Colab

In [ ]:
import os
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/tomaztc/Tomav2.git"
    REPO_DIR = "/content/Tomav2/"
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !sudo dpkg --add-architecture i386
        !sudo apt-get update -qq
        !sudo apt-get install -y wine wine64 wine32 xfoil
    os.chdir(REPO_DIR)

## Instalar pacotes pip

In [ ]:
from importlib.util import find_spec
if find_spec("numpy") is None or find_spec("pandas") is None or find_spec("matplotlib") is None \
    or find_spec("scipy") is None or find_spec("seaborn") is None:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy", "pandas", "matplotlib", "scipy", "seaborn"])

## Imports

In [ ]:
import pickle
import time
import os
from pprint import pprint
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.optimize import root_scalar, minimize, Bounds
from scipy import integrate

import airfoil_mod as am
from eulerblock import euler_mod as eb

from designTool.aerodynamics import aerodynamics
from designTool.auxiliary import atmosphere
from designTool.constants import gravity, ft2m
from designTool.standard_airplane import standard_airplane
from designTool.analyze import analyze

In [ ]:
np.random.seed(1234)
pd.options.display.float_format = "{:,.2f}".format
sns.set_theme(style="whitegrid", context="notebook")

## Funções

In [ ]:
def calcular_CL_voo(condicao, peso):
    atm = atmosphere(condicao["altitude"])
    rho = atm["density"]
    Mach = condicao["Mach"]
    V = Mach*atm["speed_of_sound"]
    CL = 2*peso / (rho*V**2*S_w)
    return CL

def aerodynamics_condicao_e_CL(airplane, condicao, CL):
    CD, CLmax, dragDict = aerodynamics(airplane, condicao["Mach"], condicao["altitude"], CL,
                                       n_engines_failed=condicao["n_engines_failed"],
                                       highlift_config=condicao["highlift_config"], lg_down=condicao["lg_down"],
                                       h_ground=condicao["h_ground"])
    return CD, CLmax, dragDict

def calcular_secao_asa(airplane, condicao, y, W_medio_cruise, log=False):
    # Geometria
    tct_w = airplane["inputs"]["tct_w"]
    tcr_w = airplane["inputs"]["tcr_w"]
    b_w = float(airplane["geometry"]["b_w"])
    cr_w = float(airplane["geometry"]["cr_w"])
    ct_w = float(airplane["geometry"]["ct_w"])
    tc_y = np.interp(y, [0, b_w/2], [tcr_w, tct_w])
    c_y = np.interp(y, [0, b_w/2], [cr_w, ct_w])

    # Aerodinâmica
    Mach = condicao["Mach"]
    altitude = condicao["altitude"]
    atm = atmosphere(altitude)
    rho = atm["density"]
    mu = atm["dyn_viscosity"]
    V_som = atm["speed_of_sound"]
    V = Mach*V_som
    Re = rho*V*c_y/mu

    # cl(y)
    # Distribuição elíptica de sustentação:
    # l(y) = l0 * sqrt(1 - (2y/b_w)^2)
    # Sustentação do perfil:
    # l(y) = q * c(y) * cl(y) (q = rho*V^2/2)
    # Sustentação total da asa: L = q*S_w*CL = W
    # 2 * integral(l(y), 0, b_w/2) = L
    # 2 * integral(l0 * sqrt(1 - (2y/b_w)^2), 0, b_w/2) = L
    # 2 * l0 * (pi*b_w/8) = L
    # l0 * pi * b_w / 4 = L
    # l0 = 4*L/(pi*b_w)
    q = rho*V**2/2
    L = W_medio_cruise
    l0 = 4*L/(np.pi*b_w)
    l_y = l0 * np.sqrt(1 - (2*y/b_w)**2)
    cl_y = l_y/(q*c_y)
    t_y = tc_y*c_y

    # Print
    if log:
        print(f"    y = {y:.2f} m")
        print(f"    tc_y = {tc_y:.1%}")
        print(f"    t_y = {(t_y*100):.2f} cm")
        print(f"    c_y = {c_y:.2f} m")
        print(f"    Re = {Re:.2e}")
        print(f"    cl_y = {cl_y:.4f}")

    return tc_y, c_y, Re, cl_y


def ajustar_tabela_latex(latex):
    latex = latex.replace(r"\toprule", "")
    latex = latex.replace(r"\midrule", r"\hline")
    latex = latex.replace(r"\bottomrule", "")
    latex = latex.replace(r"\hline \\", r"\hline")
    latex = latex.replace("%", r"\%")
    return latex

def rodar_xfoil(airfoil_path, Re):
    comandos = f"""LOAD {airfoil_path}
PPAR
T 0.45
N 200


OPER
ITER 100
v
{Re}
"""

def rodar_euler(Al, Au, alpha, Mach, plot=False): 
    # 02_single_run/single_run.py

    # Flow conditions
    gamma = 1.4 # ratio of specific heats of the fluid

    # Solver parameters
    order = 2 # 1 for 1st; 2 for 2nd
    iter = 20000 # Use 50000 for first order
    dt = 0.001 # Global time step (only used if use_local_dt==0) .Use 0.002 for first order and 0.001 for second order.
    CFL = 0.2 # Courant number used to compute local time steps (only used if use_local_dt==1)
    use_local_dt = 1 # Time stepping-strategy. 0 for global dt; 1 for local dt based on CFL.
    res_NK = 1e-4
    res_tol = 1e-6
    reinitialize = 0
    adj_funcs = ['cl_jlow', 'cd_jlow', 'cm_jlow']

    # Discretization (higher grid levels have higher number of elements)
    grid_level = 1.0

    # Reference mesh properties
    Nchord0 = 30 # Number of elements along the chord
    NJ0 = 48 # Number of elements along the normal direction
    s00 = 0.5e-2 # Reference mesh spacing

    Nchord = int((Nchord0*grid_level) + 1)
    NJ = int((NJ0*grid_level) + 1)
    s0 = NJ0/(NJ-1)*s00

    results = eb.run_cst(Al, Au, Nchord, alpha, Mach,
                        gamma, order,
                        iter, dt, CFL, use_local_dt, res_NK, res_tol,
                        reinitialize, adj_funcs, plot=plot, NJ=NJ, s0=s0)

    CL = results['CL']
    CD = results['CD']
    CM = results['CM']
    maxt = results['maxt']
    xmaxt = results['xmaxt']
    mint = results['mint']
    maxc = results['maxc']
    xmaxc = results['xmaxc']
    xx = results['distrib']['xx']
    Cp = results['distrib']['Cp']
    MachD = results['distrib']['Mach']
    grads = results['grads']

    print('Results')
    print('max thickness=',maxt)
    print('max thickness x/c=',xmaxt)
    print('max camber=',maxc)
    print('max camber x/c=',xmaxc)
    print('CL=',CL)
    print('CD=',CD)
    print('CM=',CM)
    # print('grads=',grads)

    # # Refine coordinates for xfoil export
    # x = (1-np.cos(np.linspace(0, 1, 81)*np.pi))/2
    # airfoil = am.cstfoil(Au, Al, x)
    # xf = airfoil['x_coord']
    # yf = airfoil['y_coord']
    # maxt = airfoil['max_thickness']
    # xmaxt = airfoil['x_max_thickness']
    # mint = airfoil['min_thickness']
    # maxc = airfoil['max_camber']
    # xmaxc = airfoil['x_max_camber']
    # am.export_airfoil(airfoil, title='AIRFOIL', filename='airfoil.dat')

    # # Example of how to plot Cp curve and airfoil coordinates
    # fig = plt.figure()
    # plt.subplot(311)
    # plt.plot(xx,Cp)
    # plt.ylabel('Cp')
    # plt.gca().invert_yaxis()
    # plt.subplot(312)
    # plt.plot(xx,MachD)
    # plt.ylabel('Mach')
    # plt.subplot(313)
    # plt.plot(xf,yf)
    # plt.axis('equal')
    # plt.ylabel('y/c')
    # plt.xlabel('x/c')
    # plt.tight_layout()
    # plt.show()

    return results

def exportar_airfoil(Al, Au, title='AIRFOIL', filename='airfoil.dat'):
    # Refine coordinates for xfoil export
    x = (1-np.cos(np.linspace(0, 1, 81)*np.pi))/2
    airfoil = am.cstfoil(Au, Al, x)
    am.export_airfoil(airfoil, title=title, filename=filename, close_te=False)

def encontrar_alpha_para_cl(pasta, Al, Au, Mach, cl, bracket):
    with working_directory(pasta):
        results = {}
        i = 1
        def rodar_euler_alpha(alpha):
            nonlocal results, i
            print(f"\n=== {i}. Testando alpha = {np.rad2deg(alpha):.3f}° ===\n")
            results = rodar_euler(Al, Au, alpha, Mach)
            i += 1
            return results["CL"]

        print(f"Encontrando alpha para cl = {cl:.4f}")
        sol = root_scalar(
            lambda alpha: rodar_euler_alpha(alpha) - cl,
            bracket=bracket,
            method='brentq',
            xtol=np.deg2rad(0.01)
        )
        alpha = sol.root
        print(f"\n=== alpha encontrado com {i-1} chamadas ao Euler Solver: {np.rad2deg(alpha):.2f}° ===\n")
        return alpha, results

def otimizar_airfoil(pasta, Mach, Al_0, Au_0, CLref, tref, alpha_0, alpha_min=np.deg2rad(-3), alpha_max=np.deg2rad(8), Al_lower=-1, Al_upper=-0.05, Au_lower=0.05, Au_upper=1.00):
    with working_directory(pasta):
        results = {}
        #===============================================
        # INPUTS
        # Minimum thickness allowed
        tlower = 0.005

        # Euler solver parameters
        order = 2 # 1 for 1st; 2 for 2nd
        iter = 20000 # max number of iterations (Use 50000 for first order)
        dt = 0.001 # Global time step (only used if use_local_dt==0). Use 0.002 for first order and 0.001 for second order. Reduce if you are getting NaNs or unstable solutions
        CFL = 0.2 # Courant number used to compute local time steps (only used if use_local_dt==1). Reduce if you are getting NaNs or unstable solutions
        use_local_dt = 1 # Time stepping-strategy. 0 for global dt; 1 for local dt based on CFL.
        res_NK = 1e-5 # Tolerance to switch from Runge-Kutta to Newton-Krylov solver. This value should be greater than res_tol.
        res_tol = 1e-8 # Tolerance on residual MSE to stop iterations. I do not recommend using values greater than 1e-6 (e.g. 1e-5), otherwise the numerical noise will break finite-difference estimations

        # Airfoil discretization
        Nchord = 31
        cref = 1.0 # reference chord

        # Number of airfoil CST parameters (design variables of the problem)
        Nvar = len(Al_0) # This value is used for the upper and lower skin separately (the total number of DVs is 2*Nvar)

        # Bounds for design variables
        Al_lower = [Al_lower]*Nvar
        Al_upper = [Al_upper]*Nvar
        Au_lower = [Au_lower]*Nvar
        Au_upper = [Au_upper]*Nvar

        # Select if we should use reinitialization for the first CFD analysis.
        # reinitialize = [0] does not use reinitialization for the first CFD analysis.
        # reinitialize = [1] uses reinitialization for the first CFD analysis.
        # We set it as a list so it remains accessible within the the check_point function.
        reinitialize = [0]

        #===============================================
        # EXECUTION

        # Initial airfoil design variables
        # The structure of the DV array is:
        # xx = [lower skin CST coefficients (Al); upper skin CST coefficients (Au); alpha]
        xx0 = np.hstack([Al_0, Au_0, alpha_0])

        # Initialize optimization history
        xxhist = []
        CDhist = []
        CLhist = []
        maxthist = []
        minthist = []
        gradshist = []

        # Define function to check if the desired point was already executed
        def check_point(xx, tol=1e-11, force_run=False):
            nonlocal results
            # Loop over the history to find previous design point within tolerance
            ii_closest = -1
            for ii,xx_curr in enumerate(xxhist):
                delta = np.sqrt(np.sum((xx_curr - xx)**2))
                if delta < tol:
                    print('Found previous solution')
                    print('ii:',ii)
                    print('delta:',delta)
                    ii_closest = ii
                    break

            # Get design variables
            Al = xx[:Nvar]
            Au = xx[Nvar:2*Nvar]
            alpha = xx[-1]
            
            print('Evaluating the following point')
            print('Al:',Al)
            print('Au:',Au)
            print('alpha [deg]:',alpha*180/np.pi)

            # Choose whether to run new case or use history
            if (ii_closest == -1) or force_run: # i_closest keeps the default value if no match is found
                # Call Euler solver with adjoint for each function
                results = eb.run_cst(Al, Au, Nchord, alpha, Mach,
                                    gamma=1.4, order=order,
                                    iter=iter, dt=dt, CFL=CFL, use_local_dt=use_local_dt, res_NK=res_NK,
                                    res_tol=res_tol, reinitialize=reinitialize[0], plot=False,
                                    adj_funcs=['cl_jlow','cd_jlow'])
                CL = results['CL']
                CD = results['CD']
                maxt = results['maxt']
                mint = results['mint']
                grads1 = results['grads']

                # Gather gradients w.r.t. xx
                grads = {}
                grads['CL'] = np.hstack([grads1['cl_jlow']['dAl'],
                                        grads1['cl_jlow']['dAu'],
                                        [grads1['cl_jlow']['alpha']]])
                grads['CD'] = np.hstack([grads1['cd_jlow']['dAl'],
                                        grads1['cd_jlow']['dAu'],
                                        [grads1['cd_jlow']['alpha']]])
                grads['maxt'] = np.hstack([grads1['maxt']['dAl'],
                                        grads1['maxt']['dAu'],
                                        [0.0]])
                grads['mint'] = np.hstack([grads1['mint']['dAl'],
                                        grads1['mint']['dAu'],
                                        [0.0]])

                xxhist.append(xx.copy())
                CDhist.append(CD)
                CLhist.append(CL)
                maxthist.append(maxt)
                minthist.append(mint)
                gradshist.append(grads)
                with open('opt_results.pickle','wb') as fid:
                    pickle.dump({'xxhist':xxhist,
                                'CDhist':CDhist,
                                'CLhist':CLhist,
                                'maxthist':maxthist,
                                'minthist':minthist,
                                'mach':Mach,
                                'gradshist':gradshist,
                                'Al':Al,
                                'Au':Au}, fid)

            else:
                # Gather values from previous solution
                CL = CLhist[ii_closest]
                CD = CDhist[ii_closest]
                maxt = maxthist[ii_closest]
                mint = minthist[ii_closest]
                grads = gradshist[ii_closest]

            # Print and store history
            print('evaluation:',len(CDhist))
            print('CL:',CL)
            print('CD:',CD)
            print('maxt:',maxt)
            print('mint:',mint)
            print('Al:',Al)
            print('Au:',Au)
            print('alpha [deg]:',alpha*180/np.pi)
            print('grads:',grads)
            print('')

            # Check if we need to avoid solution reinitialization if NaNs show up
            if np.isnan(CL) or np.isnan(CD):
                reinitialize[0] = 0
            else:
                reinitialize[0] = 1

            # Return design functions
            return CL, CD, maxt, mint, grads


        # Define objective function
        def objfun(xx):
            CL, CD, maxt, mint, grads = check_point(xx)
            return CD

        def objfungrad(xx):
            CL, CD, maxt, mint, grads = check_point(xx)
            return grads['CD']

        # Define constraints
        def ineqconfun(xx):
            CL, CD, maxt, mint, grads = check_point(xx)
            g1 = (maxt - tref)
            # g2 = (tlower - mint)
            g2 = (mint - tlower)
            return g1, g2

        def ineqconfungrad(xx):
            CL, CD, maxt, mint, grads = check_point(xx)
            # return grads['maxt'], -grads['mint']
            return grads['maxt'], grads['mint']

        # Define constraints
        def eqconfun(xx):
            CL, CD, maxt, mint, grads = check_point(xx)
            h1 = (CL - CLref)
            return h1

        def eqconfungrad(xx):
            CL, CD, maxt, mint, grads = check_point(xx)
            return grads['CL']

        # Run initial case
        objfun(xx0)
        reinitialize[0] = 1

        # Create list of constraints
        con1 = {'type': 'ineq',
                'fun': ineqconfun,
                'jac': ineqconfungrad}
        con2 = {'type': 'eq',
                'fun': eqconfun,
                'jac': eqconfungrad}
        cons = [con1, con2]

        # Set DV bounds
        #lb = [-Amax]*Nvar + [Amin]*Nvar + [alpha_min]
        #ub = [-Amin]*Nvar + [Amax]*Nvar + [alpha_max]
        lb = Al_lower + Au_lower + [alpha_min]
        ub = Al_upper + Au_upper + [alpha_max]
        bounds = Bounds(lb, ub, keep_feasible=True)

        # Set optimizer options.
        # Adjusted eps (finite difference size) according to expected DV bounds.
        options={'maxiter': 100, 'ftol': 1e-05, 'iprint': 1, 'disp': True}#, 'finite_diff_rel_step': None}

        # Run optimizer
        start_time = time.perf_counter()
        opt_result = minimize(
            objfun,
            x0=xx0,
            jac=objfungrad,
            constraints=cons,
            bounds=bounds,
            method='slsqp',
            options=options
        )
        elapsed = time.perf_counter() - start_time
        success = opt_result.success
        # Print results
        print('')
        print('=========================================')
        print('OPTIMIZATION RESULTS')
        print(opt_result)
        print('Optimization time: %d seconds' % elapsed)
        print('Optimization success:', success)
        print('=========================================')
        print('')
        xopt = opt_result.x

        # Run optimum point to make sure it is the last item
        # of the history file
        CL, CD, maxt, mint, grads = check_point(xopt, force_run=True)

        print('')
        print('=========================================')
        print('OPTIMUM POINT')
        print('CL:',CL)
        print('CD:',CD)
        print('maxt:',maxt)
        print('xopt:',xopt)
        print('=========================================')
        print('')

        Al = xopt[0:4]
        Au = xopt[4:8]
        alpha = xopt[8]

        return opt_result, results, elapsed, Al, Au, alpha


def plot_history(pasta):
    with working_directory(pasta):
        with open('opt_results.pickle','rb') as fid:
            hist_dict = pickle.load(fid)

        xx_hist = np.asarray(hist_dict['xxhist'])
        CD_hist = np.asarray(hist_dict['CDhist'])
        CL_hist = np.asarray(hist_dict['CLhist'])
        maxt_hist = np.asarray(hist_dict['maxthist'])
        mint_hist = np.asarray(hist_dict['minthist'])
        mach_hist = np.asarray(hist_dict['mach'])

        plt.figure()
        plt.subplot(411)
        sns.lineplot(data=xx_hist, legend=False)
        plt.ylabel(r'$x$')
        plt.subplot(412)
        sns.lineplot(data=CD_hist)
        plt.ylabel(r'$C_D$')
        plt.subplot(413)
        sns.lineplot(data=CL_hist)
        plt.ylabel(r'$C_L$')
        plt.subplot(414)
        sns.lineplot(data=maxt_hist)
        sns.lineplot(data=mint_hist)
        plt.ylabel(r'$t/c$')
        plt.xlabel('Function evaluations')
        plt.tight_layout()

        # The number of airfoil coefficients is half of xx
        # after discounting the alpha
        Nvar = (len(xx_hist[0,:])-1)//2

        Al = xx_hist[-1,:Nvar]
        Au = xx_hist[-1,Nvar:2*Nvar]
        alpha = xx_hist[-1,-1]
        xa = (1-np.cos(np.linspace(0, 1, 81)*np.pi))/2

        airfoil = am.cstfoil(Au, Al, xa)
        x = airfoil['x_coord']
        y = airfoil['y_coord']
        maxt = airfoil['max_thickness']
        xmaxt = airfoil['x_max_thickness']
        mint = airfoil['min_thickness']
        maxc = airfoil['max_camber']
        xmaxc = airfoil['x_max_camber']

        # Export airfoil coordinates
        am.export_airfoil(airfoil)

        # Read the most recent wall.dat file stored in the folder
        CL,CD,CM,xxD,Cp,MachD = eb.postprocess(x,y,alpha,mach_hist,plot=True)

        print('Results for the last airfoil in history')
        print('max thickness=',maxt)
        print('max thickness x/c=',xmaxt)
        print('max camber=',maxc)
        print('max camber x/c=',xmaxc)
        print('CL=',CL)
        print('CD=',CD)
        print('CM=',CM)

        plt.show()

@contextmanager
def working_directory(path):
    old = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)

## Atividades

### 1. Primeiramente precisamos determinar um ponto de projeto (altitude, Mach e peso) para otimizar o aerofólio. Uma vez que esse ponto for estabelecido, calcule o coeficiente de sustentação (CL) correspondente para a aeronave. Em seguida, tome a corda média aerodinâmica da asa como corda de referência (cref) e, para fins desse exercício, assuma que o coeficiente de sustentação dessa seção é igual ao CL da aeronave (clref = CL). Utilize a espessura média da aeronave da equipe para determinar o valor de espessura mínima para a otimização (t/c)ref. Utilize tais dados para preencher a Tab. 1. Mostre no relatório o procedimento de cálculo dos itens presentes na planilha.


#### Avião Tomav no cruzeiro

In [ ]:
airplane_name = "Tomav"
airplane = standard_airplane(airplane_name)
analyze(airplane)

W0 = float(airplane["thrust_matching"]["W0"])
b_w = float(airplane["geometry"]["b_w"])
S_w = airplane["inputs"]["S_w"]
clmax_w = airplane["inputs"]["clmax_w"]
Mach_cruise = airplane["inputs"]["Mach_cruise"]
altitude_cruise = airplane["inputs"]["altitude_cruise"]

CondicaoCruise = {
    "Mach": Mach_cruise,
    "altitude": altitude_cruise,
    "n_engines_failed": 0,
    "highlift_config": "clean",
    "lg_down": 0,
    "h_ground": 0,
}

print(f"Mach_cruise = {Mach_cruise:.2f}")
print(f"altitude_cruise = {altitude_cruise:.0f} m")

#### Peso médio no cruzeiro

In [ ]:
# Equação de Breguet: R = (V/TSFC)*(L/D)*ln(Wi/Wf)
# => Wf = Wi * exp(-K*R), K = 1/((V/TSFC)*(L/D))
# W_medio = W(x=R/2)
# W_medio = Wi * exp(-K*R/2)
# W_medio = Wi * sqrt(exp(-K*R))
# W_medio = Wi * sqrt(Wf/Wi)
# W_medio = sqrt(Wi * Wf) 
# => média geométrica

fases = ["start", "taxi", "takeoff", "climb", "cruise", "descent", "altcruise", "loiter", "landing"]
fracoes = [airplane["fuel_weight"]["Mf_hist"][fase] for fase in fases]
pesos = [0 for _ in range(len(fracoes) + 1)]
pesos[0] = W0
for i, f in enumerate(fracoes):
    pesos[i + 1] = pesos[i] * fracoes[i]
PesosFinais = {fase: peso for fase, peso in zip(fases, pesos[1:])}
PesosIniciais = {fase: peso for fase, peso in zip(fases, pesos[:-1])}

W_medio_cruise = (PesosIniciais["cruise"]*PesosFinais["cruise"])**0.5
print(f"W_medio_cruise = {(W_medio_cruise/gravity):.1f} kgf")

#### Aerodinâmica média no cruzeiro

In [ ]:
CL_medio_cruise = calcular_CL_voo(CondicaoCruise, W_medio_cruise)
CD_medio_cruise, CLmax_medio_cruise, dragDict_medio_cruise = aerodynamics_condicao_e_CL(airplane, CondicaoCruise, CL_medio_cruise)

atm = atmosphere(altitude_cruise)
rho = atm["density"]
V_som = atm["speed_of_sound"]

print(f"CL_medio_cruise = {CL_medio_cruise:.4f}")
print(f"CD_medio_cruise = {CD_medio_cruise:.4f}")
print(f"CLmax_medio_cruise = {CLmax_medio_cruise:.4f}")
print(f"Velocidade do som = {V_som:.2f} m/s")
print(f"Densidade do ar = {rho:.4f} kg/m³")

#### Seções da asa

In [ ]:
y_A = 0
y_B = 0.9 * (b_w/2)
y_medio = 0.5 * (b_w/2)

print("* Seção A:")
tc_A, c_A, Re_A, cl_A = calcular_secao_asa(airplane, CondicaoCruise, y_A, W_medio_cruise, log=True)
print("* Seção B:")
tc_B, c_B, Re_B, cl_B = calcular_secao_asa(airplane, CondicaoCruise, y_B, W_medio_cruise, log=True)
print("* Seção na metade:")
tc_medio, c_medio, Re_medio, cl_medio = calcular_secao_asa(airplane, CondicaoCruise, y_medio, W_medio_cruise, log=True)

#### Tabela 1

In [ ]:
tabela1 = pd.DataFrame({
    "Parâmetro": [
        r"$h$",
        r"$M_\infty$",
        r"$W$",
        r"$S_{\mathrm{ref}}$",
        r"$b_w$",
        r"$\rho_\infty$",
        r"$a_\infty$",
        r"$y_{A}$",
        r"$c_{A}$",
        r"$c_{l,A}$",
        r"$Re_{A}$",
        r"$(t/c)_{A}$",
        r"$y_{B}$",
        r"$c_{B}$",
        r"$c_{l,B}$",
        r"$Re_{B}$",
        r"$(t/c)_{B}$",
        r"$y_{m}$",
        r"$c_{m}$",
        r"$c_{l,m}$",
        r"$Re_{m}$",
        r"$(t/c)_{m}$"
    ],
    "Valor": [
        f"{(altitude_cruise/ft2m):.0f} ft",
        f"{Mach_cruise:.2f}",
        f"{W_medio_cruise/gravity:.1f} kgf",
        rf"{S_w:.1f} $\mathrm{{m^2}}$",
        f"{b_w:.2f} m",
        rf"{rho:.3f} $\mathrm{{kg/m^3}}$",
        rf"{V_som:.1f} $\mathrm{{m/s}}$",
        f"{y_A:.2f} m",
        f"{c_A:.2f} m",
        f"{cl_A:.4f}",
        f"{Re_A:.2e}",
        f"{tc_A:.1%}",
        f"{y_B:.2f} m",
        f"{c_B:.2f} m",
        f"{cl_B:.4f}",
        f"{Re_B:.2e}",
        f"{tc_B:.1%}",
        f"{y_medio:.2f} m",
        f"{c_medio:.2f} m",
        f"{cl_medio:.4f}",
        f"{Re_medio:.2e}",
        f"{tc_medio:.1%}"
    ],
    "Descrição": [
        "Altitude do ponto de projeto",
        "Mach do ponto de projeto",
        "Peso da aeronave no ponto de projeto",
        "Área de referência da aeronave",
        "Envergadura da asa",
        "Densidade do ar no ponto de projeto",
        r"Velocidade do som no ponto de projeto \\ \hline",
        "Posição da Seção A",
        "Corda da Seção A",
        "Coeficiente de sustentação da Seção A",
        "Reynolds para a Seção A",
        r"Espessura relativa da Seção A \\ \hline",
        "Posição da Seção B",
        "Corda da Seção B",
        "Coeficiente de sustentação da Seção B",
        "Reynolds para a Seção B",
        "Espessura relativa da Seção B",
        "Posição da Seção Média",
        "Corda da Seção Média",
        "Coeficiente de sustentação da Seção Média",
        "Reynolds para a Seção Média",
        "Espessura relativa da Seção Média"
    ]
})

print("Tabela 1:")
display(tabela1)

#### LaTeX

In [ ]:
latex1 = tabela1.to_latex(
    index=False,
    escape=False,
    na_rep="",
    caption="Dados para as seções de referência",
    label="tab:dados_secoes",
    column_format=r"c|c|l"
)
latex1 = ajustar_tabela_latex(latex1)
print(latex1)

### 2. Utilize os scripts da pasta 02_single_run para determinar qual o ângulo de ataque (variável alpha da linha 19) faz com que o perfil NACA 1411 atinja o cl de projeto. Lembre de atualizar também o número de Mach.

In [ ]:
Al_NACA1411 = [-0.1489439, -0.10330027, -0.10305128, -0.10514982]
Au_NACA1411 = [0.16146332, 0.18349204, 0.14126241, 0.18194397]
bracket = (np.deg2rad(-3), np.deg2rad(7))

#### Seção A

In [ ]:
alpha_NACA_A, results_NACA_A = encontrar_alpha_para_cl("Lab3/NACA_A", Al_NACA1411, Au_NACA1411, Mach_cruise, cl_A, bracket)

print("Resultados A:")
pprint(results_NACA_A)
# CL = results['CL']
# CD = results['CD']
# CM = results['CM']
# maxt = results['maxt']
# xmaxt = results['xmaxt']
# mint = results['mint']
# maxc = results['maxc']
# xmaxc = results['xmaxc']
# xx = results['distrib']['xx']
# Cp = results['distrib']['Cp']
# MachD = results['distrib']['Mach']
# grads = results['grads']

print(f"\nalpha_A = {np.rad2deg(alpha_NACA_A):.2f}°")
print(f"cl_A alvo = {cl_A:.4f}")
print(f"cl_A = {results_NACA_A["CL"]:.4f}")

#### Seção B

In [ ]:
alpha_NACA_B, results_NACA_B = encontrar_alpha_para_cl("Lab3/NACA_B", Al_NACA1411, Au_NACA1411, Mach_cruise, cl_B, bracket)

print("Resultados B:")
pprint(results_NACA_B)

print(f"\nalpha_B = {np.rad2deg(alpha_NACA_B):.2f}°")
print(f"cl_B alvo = {cl_B:.4f}")
print(f"cl_B = {results_NACA_B["CL"]:.4f}")

### 3. Utilize os scripts da pasta 03_optimize para obter um aerofólio otimizado para as condições de projeto. O ponto de partida da otimização será o aerofólio NACA 1411. Utilize como batentes para parâmetros do CST os valores Al,upper = −0.05 e Au,lower = 0.05. Preencha a Tab. 2 com os resultados da otimização. ATENÇÃO: Cada otimização pode demorar horas, por isso guarde bem o resultado.


Otimização:

min cd

w.r.t. CST coefficients(Al, Au), α

s.t. cl = clref

clmax ≥ clmax_w

(t/c) ≥ (t/c)ref

Al,lower ⪯ Al ⪯ Al,upper

Au,lower ⪯ Au ⪯ Au,upper

In [ ]:
clmax_w = airplane["inputs"]["clmax_w"]

#### Seção A

In [ ]:
opt_result_A, results_A, elapsed_A, Al_A, Au_A, alpha_A = otimizar_airfoil("Lab3/airfoil_A", Mach_cruise, Al_NACA1411, Au_NACA1411, cl_A, tc_A, alpha_NACA_A)
print("Resultados A:")
pprint(opt_result_A)
pprint(results_A)

print(f"\nAl_A = {Al_A}")
print(f"Au_A = {Au_A}")
print(f"alpha_A = {np.rad2deg(alpha_A):.2f}°")

#### Seção B

In [ ]:
opt_result_B, results_B, elapsed_B, Al_B, Au_B, alpha_B = otimizar_airfoil("Lab3/airfoil_B", Mach_cruise, Al_NACA1411, Au_NACA1411, cl_B, tc_B, alpha_NACA_B)
print("\nResultados B:")
pprint(opt_result_B)
pprint(results_B)

print(f"\nAl_B = {Al_B}")
print(f"Au_B = {Au_B}")
print(f"alpha_B = {np.rad2deg(alpha_B):.2f}°")

#### Tabela 2

In [ ]:
tabela2 = pd.DataFrame({
    "Parâmetro": [
        r"$A_{l1}$",
        r"$A_{l2}$",
        r"$A_{l3}$",
        r"$A_{l4}$",
        r"$A_{u1}$",
        r"$A_{u2}$",
        r"$A_{u3}$",
        r"$A_{u4}$",
        r"$\alpha_A$",
        r"$c_{l,A}$",
        r"$c_{d,A}$",
        r"$c_{m,A}$",
        r"$\alpha_B$",
        r"$c_{l,B}$",
        r"$c_{d,B}$",
        r"$c_{m,B}$",
        r"$(t/c)_{\max}$",
        r"$x_{t/c,\max}$",
        r"$(h/c)_{\max}$",
        r"$x_{h/c,\max}$"
    ],

    "NACA 1411": [
        Al_NACA1411[0],
        Al_NACA1411[1],
        Al_NACA1411[2],
        Al_NACA1411[3],
        Au_NACA1411[0],
        Au_NACA1411[1],
        Au_NACA1411[2],
        Au_NACA1411[3],
        np.rad2deg(alpha_NACA_A),
        results_NACA_A["CL"],
        results_NACA_A["CD"],
        results_NACA_A["CM"],
        np.rad2deg(alpha_NACA_B),
        results_NACA_B["CL"],
        results_NACA_B["CD"],
        results_NACA_B["CM"],
        results_NACA_A["maxt"],
        results_NACA_A["xmaxt"],
        results_NACA_A["maxc"],
        results_NACA_A["xmaxc"]
    ],

    r"$Aerofólio A$": [
        Al_A[0],
        Al_A[1],
        Al_A[2],
        Al_A[3],
        Au_A[0],
        Au_A[1],
        Au_A[2],
        Au_A[3],
        np.rad2deg(alpha_A),
        results_A["CL"],
        results_A["CD"],
        results_A["CM"],
        "",
        "",
        "",
        "",
        results_A["maxt"],
        results_A["xmaxt"],
        results_A["maxc"],
        results_A["xmaxc"]
    ],

    r"$Aerofólio B$": [
        Al_B[0],
        Al_B[1],
        Al_B[2],
        Al_B[3],
        Au_B[0],
        Au_B[1],
        Au_B[2],
        Au_B[3],
        "",
        "",
        "",
        "",
        np.rad2deg(alpha_B),
        results_B["CL"],
        results_B["CD"],
        results_B["CM"],
        results_B["maxt"],
        results_B["xmaxt"],
        results_B["maxc"],
        results_B["xmaxc"]
    ],

    "Descrição": [
        "Parâmetro CST do intradorso",
        "Parâmetro CST do intradorso",
        "Parâmetro CST do intradorso",
        "Parâmetro CST do intradorso",
        "Parâmetro CST do extradorso",
        "Parâmetro CST do extradorso",
        "Parâmetro CST do extradorso",
        "Parâmetro CST do extradorso",
        r"Ângulo de ataque [$^\circ$]" + " (Seção A)",
        "Coeficiente de sustentação (Seção A)",
        "Coeficiente de arrasto (Seção A)",
        "Coeficiente de momento (Seção A)",
        "Espessura máxima (Seção B)",
        "Posição da espessura máxima (Seção B)",
        "Arqueamento máximo (Seção B)",
        "Posição do arqueamento máximo (Seção B)"
    ]
})

print("Tabela 2:")
display(tabela2)

#### LaTeX

In [ ]:
latex2 = tabela2.to_latex(
    index=False,
    escape=False,
    na_rep="",
    float_format="%.5f",
    caption="Dados dos aerofólios otimizados",
    label="tab:aerofolios_otimizados",
    column_format=r"c|c|c|c|l"
)
latex2 = ajustar_tabela_latex(latex2)
print(latex2)

### 4. Plote o histórico de convergência de cada otimização (pode usar o script plot_history.py) e discuta quanto foi o aprimoramento obtido pelo otimizador. Indique também o tempo total de otimização e como as restrições afetaram o resultado.


In [ ]:
print("Tempo A: %.2f s" % elapsed_A)
print("Tempo B: %.2f s" % elapsed_B)

### 5. Desenhe 1) gráficos sobrepondo a geometria, 2) gráficos sobrepondo curvas de Cp ao longo da corda e 3) gráficos sobrepondo curvas de Mach ao longo da corda dos aerofólios da Tab. 2 na condição de projeto transônica. O script da pasta 02_single_run pode ser útil para isso.

### 6. Use os gráficos gerados para explicar a abordagem aplicada pelo otimizador para chegar na configuração ótima.

### 7. Adapte o script 02_single_run.py para gerar uma polar de arrasto para o perfil otimizado na condição transônica. Para isso, você precisará simular o perfil para uma sequência de ângulos de ataque e depois plotar os valores de cl e cd. Sugiro usar um intervalo de -2/+2 graus em relação ao ângulo de ataque do ponto ótimo. Lembre-se de destacar o ponto de projeto (cl da otimização) na polar. Sobreponha no mesmo gráfico uma curva correspondente ao aerofólio transônico RAE2822 (os dados desse aerofólio estão no arquivo single_run.py). Discuta se a polar do aerofólio otimizado tem alguma peculiaridade e se isso é bom ou ruim para o projeto de uma aeronave.

### 8. Exporte os aerofólios iniciais e finais da otimização para o formato Xfoil e gere curvas cl × α, cl × cd e cm × α para os dois aerofólios em regime subsônico (pode desconsiderar o número de Mach para essa análise). Utilize a condição de decolagem da aeronave para determinar o número de Reynolds para essa simulação (lembre de indicar o número de Reynolds no relatório). Sobreponha as curvas dos dois aerofólios em cada plot e discuta as vantagens e desvantagens dos perfis no regime analisado.

### 9. Verifiquem a possibilidade de alterar a definição da otimização para obter um aerofólio de melhor característica no regime subsônico.

### 10. Quem estiver com o Paraview (https://www.paraview.org/download/) instalado no computador pode observar a solução no domínio carregando o arquivo paraview_state_windows.py com a opção “Load State”. Esse arquivo sempre carregará o arquivo solution.vtk da pasta 02_single_run.